# 🌞 YOLO Solar Panel Detection & Classification

## 📑 Índice de Conteúdo

- **Seção 0**: Imports e Configuração (Utilities)
- **Seção 1**: Carregamento de Dados (Lacuna Solar Survey)
- **Seção 2**: Preprocessamento & Data Augmentation
- **Seção 3**: YOLOv8 Setup & Configuration
- **Seção 4**: Treinamento do Modelo YOLO
- **Seção 5**: Detecção & Inferência
- **Seção 6**: Classificação (Transfer Learning)
- **Seção 7**: Estimativa de Potência
- **Seção 8**: Visualização & Relatórios
- **Seção 9**: Integração de Dados Existentes
- **Seção 10**: Processamento em Batch

## ⚡ Quick Start

1. Todos os pacotes devem estar instalados (`requirements.txt`)
2. Execute as células **na ordem** do índice
3. Outputs salvos em `output_detection/`
4. Modelos treinados em `modelos/`

**Versão**: 1.0 | **Data**: Jan 2026 | **Status**: ✅ Production-Ready

In [1]:
"""
YOLO SOLAR PANEL DETECTION & CLASSIFICATION
============================================
Sistema completo para detecção, classificação e estimativa de potência de painéis solares
utilizando YOLOv8, Transfer Learning e Deep Learning.
"""

# ============= IMPORTS =============

# Core Libraries
import sys
import os
import warnings
from pathlib import Path
from typing import Tuple, Optional, Dict, List

# Data Science & ML
import numpy as np
import pandas as pd
import cv2
from PIL import Image

# ML Models
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============= ENVIRONMENT SETUP =============

print(f"🔧 Configurando ambiente...")
print(f"  Python: {sys.executable}")
print(f"  Versão: {sys.version.split()[0]}")
print(f"  Tipo: {'Conda' if 'conda' in sys.executable else 'Virtual Environment (venv)'}")

# ============= PATH CONFIGURATION =============

BASE_PATH = Path('.')
DATA_PATH = BASE_PATH / 'data' / 'solar_panel'
MODELS_PATH = BASE_PATH / 'modelos'
OUTPUT_PATH = BASE_PATH / 'output_detection'

# Criar diretórios
for path in [OUTPUT_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Ambiente pronto!")
print(f"  - Data: {DATA_PATH.absolute()}")
print(f"  - Modelos: {MODELS_PATH.absolute()}")
print(f"  - Output: {OUTPUT_PATH.absolute()}")

🔧 Configurando ambiente...
  Python: /home/jovyan/yolo_venv/bin/python
  Versão: 3.11.6
  Tipo: Virtual Environment (venv)

✓ Ambiente pronto!
  - Data: /home/jovyan/work/data/solar_panel
  - Modelos: /home/jovyan/work/modelos
  - Output: /home/jovyan/work/output_detection


In [2]:
# ============= UTILITY FUNCTIONS =============

def load_image(image_path: Path) -> Optional[np.ndarray]:
    """Carregar imagem com tratamento de erro"""
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            print(f"⚠️ Não foi possível carregar: {image_path}")
            return None
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"❌ Erro ao carregar {image_path}: {e}")
        return None


def display_image(img: np.ndarray, title: str = "Image", figsize: Tuple[int, int] = (10, 8)):
    """Exibir imagem com matplotlib"""
    plt.figure(figsize=figsize)
    if len(img.shape) == 3 and img.shape[2] == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def ensure_directory(path: Path) -> Path:
    """Garantir que diretório existe"""
    path.mkdir(parents=True, exist_ok=True)
    return path


def print_section(title: str):
    """Imprimir separador de seção"""
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}\n")


# 📋 Fluxo Executivo

## 🎯 Objetivos

1. **Carregamento de Dados**: Integração com dataset Lacuna Solar Survey
2. **Detecção com YOLO**: Localizar painéis solares em imagens de satélite
3. **Classificação**: Classificar painéis e áreas
4. **Estimativa de Potência**: Calcular potência estimada baseada em área
5. **Visualização**: Gerar mapas e relatórios

## 📊 Pipeline Principal

```
[Dados Brutos] → [Preprocessing] → [YOLO Detecção] → [Classificação] → [Potência] → [Relatório]
```

## 🚀 Como Usar

- Execute **célula por célula** de cima para baixo
- Cada seção é independente e documentada
- Outputs são salvos em `output_detection/`

In [3]:
print_section("SEÇÃO ESPECIAL: CONVERTER DATASET LACUNA → YOLO")

# ============================================================================
# Converter dataset Lacuna Solar Survey para formato YOLO
# ============================================================================

def polygon_to_bbox(polygon_str):
    """
    Converte string de polígono para bounding box normalizado YOLO
    
    Args:
        polygon_str: String com formato "[(x1, y1), (x2, y2), ...]"
    
    Returns:
        Dict com coordenadas absolutas do bbox
    """
    import ast
    try:
        coords = ast.literal_eval(polygon_str)
        if not coords:
            return None
        
        # Extrair x e y
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        
        # Calcular bbox
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        
        return {
            'x_min': x_min,
            'y_min': y_min,
            'x_max': x_max,
            'y_max': y_max,
            'width': x_max - x_min,
            'height': y_max - y_min
        }
    except:
        return None


def prepare_lacuna_to_yolo(
    csv_path,
    images_src_dir,
    output_yolo_dir='./data/solar_panels_detection_dataset',
    train_ratio=0.7,
    val_ratio=0.15,
    image_size=(2560, 1920)
):
    """
    Converte dataset Lacuna Solar Survey para estrutura YOLO.
    
    Processa CSV com polígonos e converte para labels YOLO normalizados.
    
    Args:
        csv_path: Caminho do CSV (train.csv)
        images_src_dir: Diretório com imagens originais
        output_yolo_dir: Diretório de saída estruturado para YOLO
        train_ratio: Proporção treino (padrão 0.7)
        val_ratio: Proporção validação (padrão 0.15)
        image_size: Tamanho das imagens (largura, altura) para normalização
    
    Returns:
        Dict com estatísticas de conversão
    """
    import ast
    import shutil
    from sklearn.model_selection import train_test_split
    
    output_path = Path(output_yolo_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Criar estrutura YOLO
    for split in ['train', 'val', 'test']:
        (output_path / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_path / 'labels' / split).mkdir(parents=True, exist_ok=True)
    
    print(f"📖 Lendo CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"✓ Total de linhas: {len(df)}")
    
    # Agrupar por imagem (uma imagem pode ter múltiplos painéis)
    images_data = {}
    for idx, row in df.iterrows():
        img_id = row['ID']
        if img_id not in images_data:
            images_data[img_id] = []
        
        bbox = polygon_to_bbox(row['polygon'])
        if bbox:
            images_data[img_id].append(bbox)
    
    print(f"✓ Total de imagens únicas com painéis: {len(images_data)}")
    
    # Copiar imagens e gerar labels
    images_found = 0
    images_copied = 0
    labels_created = 0
    
    images_src = Path(images_src_dir)
    img_extensions = ['jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG']
    
    for img_id, bboxes in images_data.items():
        # Procurar imagem (pode ter várias extensões)
        img_file = None
        for ext in img_extensions:
            candidate = images_src / f"{img_id}.{ext}"
            if candidate.exists():
                img_file = candidate
                images_found += 1
                break
        
        if not img_file:
            continue
        
        # Copiar imagem para train (será reorganizada depois)
        dst_img = output_path / 'images' / 'train' / img_file.name
        shutil.copy2(img_file, dst_img)
        images_copied += 1
        
        # Gerar label YOLO (classe 0 = painel solar)
        label_file = output_path / 'labels' / 'train' / f"{img_id}.txt"
        
        with open(label_file, 'w') as f:
            for bbox in bboxes:
                # Normalizar para YOLO (0-1)
                center_x = (bbox['x_min'] + bbox['x_max']) / 2 / image_size[0]
                center_y = (bbox['y_min'] + bbox['y_max']) / 2 / image_size[1]
                width = bbox['width'] / image_size[0]
                height = bbox['height'] / image_size[1]
                
                # Clipar entre 0 e 1
                center_x = max(0, min(1, center_x))
                center_y = max(0, min(1, center_y))
                width = max(0, min(1, width))
                height = max(0, min(1, height))
                
                # Classe 0 = painel solar
                f.write(f"0 {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}\n")
                labels_created += 1
    
    print(f"\n✅ Imagens encontradas: {images_found}")
    print(f"✅ Imagens copiadas: {images_copied}")
    print(f"✅ Labels criados: {labels_created}")
    
    # Reorganizar em train/val/test
    if images_copied == 0:
        print("⚠️ Nenhuma imagem foi copiada!")
        return {'status': 'failed', 'images': 0}
    
    print(f"\n📊 Reorganizando em train/val/test...")
    all_imgs = list((output_path / 'images' / 'train').glob('*'))
    
    # Split
    train_imgs, temp = train_test_split(all_imgs, test_size=(1-train_ratio), random_state=42)
    val_ratio_adj = val_ratio / (1 - train_ratio)
    val_imgs, test_imgs = train_test_split(temp, test_size=(1-val_ratio_adj), random_state=42)
    
    splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}
    
    # Mover arquivos para split correto
    for split_name, imgs in splits.items():
        for img_file in imgs:
            if split_name != 'train':
                # Mover imagem
                dst = output_path / 'images' / split_name / img_file.name
                img_file.rename(dst)
                
                # Mover label
                label_src = output_path / 'labels' / 'train' / f"{img_file.stem}.txt"
                label_dst = output_path / 'labels' / split_name / f"{img_file.stem}.txt"
                if label_src.exists():
                    label_src.rename(label_dst)
    
    print(f"\n✅ Dataset Lacuna convertido para YOLO:")
    print(f"   📁 Treino: {len(train_imgs)} imagens")
    print(f"   📁 Validação: {len(val_imgs)} imagens")
    print(f"   📁 Teste: {len(test_imgs)} imagens")
    print(f"   💾 Salvo em: {output_path}\n")
    
    return {
        'status': 'success',
        'images_total': images_copied,
        'labels_total': labels_created,
        'train': len(train_imgs),
        'val': len(val_imgs),
        'test': len(test_imgs)
    }


# ============================================================================
# EXECUTAR CONVERSÃO - Apenas se dataset estiver vazio
# ============================================================================

def is_dataset_empty(dataset_dir: str = './data/solar_panels_detection_dataset') -> bool:
    """
    Verifica se o dataset YOLO está vazio (sem imagens nas pastas).
    
    Args:
        dataset_dir: Caminho do dataset YOLO
    
    Returns:
        True se dataset está vazio, False caso contrário
    """
    dataset_path = Path(dataset_dir)
    
    if not dataset_path.exists():
        return True
    
    # Verificar se há imagens nos diretórios train/val/test
    images_dir = dataset_path / 'images'
    if not images_dir.exists():
        return True
    
    total_images = 0
    for split in ['train', 'val', 'test']:
        split_dir = images_dir / split
        if split_dir.exists():
            total_images += len(list(split_dir.glob('*')))
    
    return total_images == 0


# Verificar se dataset está vazio
dataset_empty = is_dataset_empty()

if dataset_empty:
    print("\n🚀 Convertendo dataset Lacuna Solar Survey para formato YOLO...\n")
    
    result = prepare_lacuna_to_yolo(
        csv_path='./data/lacuna-solar-survey-zindi-2/train.csv',
        images_src_dir='./data/lacuna-solar-survey-zindi-2/images',
        output_yolo_dir='./data/solar_panels_detection_dataset'
    )
    
    if result['status'] == 'success':
        print(f"✅ SUCESSO! Dataset pronto para treinamento YOLO\n")
        print(f"📊 Estatísticas:")
        print(f"   • Total de imagens: {result['images_total']}")
        print(f"   • Total de labels (painéis): {result['labels_total']}")
        print(f"   • Treino: {result['train']}")
        print(f"   • Validação: {result['val']}")
        print(f"   • Teste: {result['test']}")
    else:
        print(f"\n❌ Erro na conversão: {result}")
else:
    print("⚠️ Dataset solar_panels_detection_dataset já existe e possui imagens!")
    print("   Pulando conversão (para reconverter, delete o diretório)")
    
    # Mostrar estatísticas do dataset existente
    dataset_path = Path('./data/solar_panels_detection_dataset')
    images_dir = dataset_path / 'images'
    labels_dir = dataset_path / 'labels'
    
    print("\n📊 Dataset existente:")
    for split in ['train', 'val', 'test']:
        img_count = len(list((images_dir / split).glob('*'))) if (images_dir / split).exists() else 0
        label_count = len(list((labels_dir / split).glob('*'))) if (labels_dir / split).exists() else 0
        print(f"   • {split.capitalize()}: {img_count} imagens, {label_count} labels")

print("\n✓ Função de conversão Lacuna → YOLO definida")



  SEÇÃO ESPECIAL: CONVERTER DATASET LACUNA → YOLO

⚠️ Dataset solar_panels_detection_dataset já existe e possui imagens!
   Pulando conversão (para reconverter, delete o diretório)

📊 Dataset existente:
   • Train: 2318 imagens, 2318 labels
   • Val: 496 imagens, 496 labels
   • Test: 498 imagens, 498 labels

✓ Função de conversão Lacuna → YOLO definida


In [4]:
# ============================================================================
# VALIDAR DATASET CRIADO (Célula 5)
# ============================================================================

print("\n" + "="*70)
print("✓ VALIDANDO DATASET CRIADO NA CÉLULA 5")
print("="*70 + "\n")

dataset_path = Path('./data/solar_panels_detection_dataset')

if not dataset_path.exists():
    print("⚠️  Dataset não foi criado. Execute a célula 5 primeiro!")
else:
    # Validar estrutura
    print("📁 Estrutura do Dataset:")
    for split in ['train', 'val', 'test']:
        images_path = dataset_path / 'images' / split
        labels_path = dataset_path / 'labels' / split
        
        img_count = len(list(images_path.glob('*'))) if images_path.exists() else 0
        label_count = len(list(labels_path.glob('*'))) if labels_path.exists() else 0
        
        print(f"\n  📊 {split.upper()}:")
        print(f"     • Imagens: {img_count}")
        print(f"     • Labels:  {label_count}")
        
        if img_count != label_count:
            print(f"     ⚠️  AVISO: Número de imagens ≠ número de labels!")
    
    # Mostrar total
    total_images = sum([len(list((dataset_path / 'images' / split).glob('*'))) 
                        for split in ['train', 'val', 'test']])
    total_labels = sum([len(list((dataset_path / 'labels' / split).glob('*'))) 
                        for split in ['train', 'val', 'test']])
    
    print(f"\n  ✅ TOTAL: {total_images} imagens, {total_labels} labels")
    
    if total_images == 0:
        print("\n  ❌ Dataset está vazio! Verifique a célula 5.")
    else:
        print("\n  ✅ Dataset pronto para treinamento YOLO!")
        
    print("\n" + "="*70)


✓ VALIDANDO DATASET CRIADO NA CÉLULA 5

📁 Estrutura do Dataset:

  📊 TRAIN:
     • Imagens: 2318
     • Labels:  2318

  📊 VAL:
     • Imagens: 496
     • Labels:  496

  📊 TEST:
     • Imagens: 498
     • Labels:  498

  ✅ TOTAL: 3312 imagens, 3312 labels

  ✅ Dataset pronto para treinamento YOLO!



In [5]:
# ============================================================================
# Nota: Treinamento movido para Seção 1 (Célula 8)
# Execute a Célula 8 para treinar o modelo com auto-detecção de GPU/CPU
# ============================================================================

print("\n✓ Seção de treinamento removida daqui")
print("   Execute a Célula 8 (SEÇÃO 1) para treinar o modelo")
print("   Aquela célula tem auto-detecção de GPU/CPU\n")


✓ Seção de treinamento removida daqui
   Execute a Célula 8 (SEÇÃO 1) para treinar o modelo
   Aquela célula tem auto-detecção de GPU/CPU



In [6]:
print_section("SEÇÃO 1: DATASET & MODEL PREPARATION")

# ============================================================================
# 1.1 - Preparação do Dataset
# ============================================================================

def prepare_yolo_dataset(
    images_dir: str, 
    labels_dir: str, 
    output_dir: str,
    train_ratio: float = 0.7, 
    val_ratio: float = 0.15
) -> str:
    """
    Prepara dataset estruturado para treinamento YOLO.
    
    Cria a estrutura: output_dir/images/{train,val,test} e labels/{train,val,test}
    
    Args:
        images_dir: Diretório com imagens (jpg, png)
        labels_dir: Diretório com labels YOLO (.txt)
        output_dir: Diretório de saída estruturado
        train_ratio: Proporção treino (padrão: 0.7)
        val_ratio: Proporção validação (padrão: 0.15)
    
    Returns:
        Caminho do diretório de saída
    
    Example:
        >>> dataset_dir = prepare_yolo_dataset(
        ...     images_dir='./data/images',
        ...     labels_dir='./data/labels',
        ...     output_dir='./data/yolo_dataset'
        ... )
    """
    import shutil
    from sklearn.model_selection import train_test_split
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Criar estrutura de diretórios
    for split in ['train', 'val', 'test']:
        (output_path / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_path / 'labels' / split).mkdir(parents=True, exist_ok=True)
    
    # Listar imagens
    images_path = Path(images_dir)
    image_files = sorted(images_path.glob('*.jpg')) + sorted(images_path.glob('*.png'))
    
    if not image_files:
        print(f"⚠️ Nenhuma imagem encontrada em {images_dir}")
        return str(output_path)
    
    print(f"📊 Total de imagens encontradas: {len(image_files)}")
    
    # Split dataset preservando a proporção teste
    train_imgs, temp_imgs = train_test_split(
        image_files, test_size=(1-train_ratio), random_state=42
    )
    val_ratio_adjusted = val_ratio / (1 - train_ratio)
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=(1-val_ratio_adjusted), random_state=42
    )
    
    splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}
    
    # Copiar arquivos para estrutura YOLO
    for split, files in splits.items():
        for img_file in files:
            # Copiar imagem
            output_img = output_path / 'images' / split / img_file.name
            shutil.copy2(img_file, output_img)
            
            # Copiar label se existir
            label_file = Path(labels_dir) / f"{img_file.stem}.txt"
            if label_file.exists():
                output_label = output_path / 'labels' / split / f"{img_file.stem}.txt"
                shutil.copy2(label_file, output_label)
    
    print(f"\n✓ Dataset preparado com sucesso:")
    print(f"   📁 Treino: {len(train_imgs)} imagens")
    print(f"   📁 Validação: {len(val_imgs)} imagens")
    print(f"   📁 Teste: {len(test_imgs)} imagens")
    print(f"   💾 Salvo em: {output_path}\n")
    
    return str(output_path)


# ============================================================================
# 1.2 - Configuração YOLO
# ============================================================================

def create_yolo_config(dataset_path: str, config_file: str = 'data.yaml') -> str:
    """
    Cria arquivo de configuração YAML para YOLOv8.
    
    Args:
        dataset_path: Caminho raiz do dataset estruturado
        config_file: Nome do arquivo de configuração
    
    Returns:
        Caminho do arquivo de configuração criado
    """
    config_content = f"""# YOLO Dataset Configuration
path: {dataset_path}
train: images/train
val: images/val
test: images/test

# Classes
nc: 1
names: ['solar_panel']
"""
    
    with open(config_file, 'w') as f:
        f.write(config_content)
    
    print(f"✓ Arquivo de configuração criado: {config_file}")
    return config_file


# ============================================================================
# 1.3 - Treinamento do Modelo
# ============================================================================

def train_yolo_model(
    config_path: str,
    model_size: str = 'm',
    epochs: int = 100,
    imgsz: int = 640,
    batch_size: int = 16,
    patience: int = 20,
    device = None  # Auto-detect (0 for GPU, 'cpu' if not available)
) -> Tuple[YOLO, Dict]:
    """
    Treina modelo YOLOv8 para detecção de painéis solares.
    
    Detecta automaticamente GPU (CUDA) e usa CPU como fallback.
    
    Args:
        config_path: Caminho do arquivo data.yaml
        model_size: Tamanho do modelo ('n', 's', 'm', 'l', 'x')
        epochs: Número de épocas de treinamento
        imgsz: Tamanho da imagem de entrada
        batch_size: Tamanho do batch
        patience: Paciência para early stopping
        device: Device a usar ('cpu', 0, 1, etc). Se None, detecta automaticamente.
    
    Returns:
        Tupla (modelo treinado, histórico de resultados)
    """
    # Detectar dispositivo automaticamente se não especificado
    if device is None:
        import torch
        device = 0 if torch.cuda.is_available() else 'cpu'
    
    print(f"\n🚀 Iniciando treinamento YOLOv8{model_size}...")
    device_name = "GPU (CUDA)" if isinstance(device, int) else "CPU"
    print(f"   Device: {device_name}")
    print(f"   Epochs: {epochs} | Batch: {batch_size} | Imgsz: {imgsz}")
    
    # Carregar modelo base
    model = YOLO(f'yolov8{model_size}.pt')
    
    # Treinar com augmentação
    results = model.train(
        data=config_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch_size,
        patience=patience,
        device=device,
        project='solar_panel_detection',
        name='yolov8_solar',
        save=True,
        verbose=True,
        # Augmentação de dados
        augment=True,
        mosaic=1.0,
        flipud=0.5,
        fliplr=0.5,
        degrees=10,
        translate=0.1,
        scale=0.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4
    )
    
    print(f"\n✓ Treinamento concluído!")
    return model, results

# Exemplo de uso:
# dataset_dir = prepare_yolo_dataset(
#     images_dir=str(DATA_PATH / 'images'),
#     labels_dir=str(DATA_PATH / 'labels'),
#     output_dir=str(OUTPUT_PATH / 'yolo_dataset')
# )
# config_path = create_yolo_config(dataset_dir)
# model, results = train_yolo_model(config_path, model_size='m', epochs=100)

print("✓ Funções de preparação e treinamento definidas")



  SEÇÃO 1: DATASET & MODEL PREPARATION

✓ Funções de preparação e treinamento definidas


In [7]:
# ============================================================================
# 1.4 - EXECUTAR TREINAMENTO (Célula dentro da Seção 1)
# ============================================================================

print("\n" + "="*70)
print("🚀 TREINANDO MODELO YOLO V8 - SOLAR PANEL DETECTION")
print("="*70 + "\n")

# Verificar se dataset existe
dataset_path = Path('./data/solar_panels_detection_dataset')
total_images = sum([len(list((dataset_path / 'images' / split).glob('*'))) 
                    for split in ['train', 'val', 'test'] if (dataset_path / 'images' / split).exists()])

if total_images == 0:
    print("❌ ERRO: Dataset está vazio!")
    print("   Execute a Célula 5 primeiro para converter o dataset Lacuna")
    print("   Depois execute a Célula 5.5 para validar")
else:
    print(f"✅ Dataset pronto: {total_images} imagens encontradas\n")
    
    # Criar arquivo de configuração YAML
    print("1️⃣ Criando arquivo de configuração YAML...")
    
    yaml_content = """# YOLO Dataset Configuration for Solar Panels Detection
path: ./data/solar_panels_detection_dataset
train: images/train
val: images/val
test: images/test

# Classes
nc: 1
names: ['solar_panel']
"""
    
    config_path = Path('solar_panels_data.yaml')
    with open(config_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"   ✅ Arquivo criado: {config_path.resolve()}\n")
    
    # Treinar o modelo
    print("2️⃣ Iniciando treinamento YOLOv8m...")
    
    # Detectar dispositivo automaticamente
    import torch
    cuda_available = torch.cuda.is_available()
    device_to_use = 0 if cuda_available else 'cpu'
    batch_size_to_use = 16 if cuda_available else 8  # Menor batch para CPU
    
    device_name = "GPU (CUDA)" if cuda_available else "CPU"
    print(f"   📊 Configuração:")
    print(f"      • Device: {device_name}")
    print(f"      • Epochs: 100")
    print(f"      • Batch size: {batch_size_to_use}")
    print(f"      • Tamanho de imagem: 640x640")
    print(f"      • Early stopping: 20 epochs\n")
    
    if not cuda_available:
        print("   ⚠️  GPU não disponível - usando CPU (treinamento será mais lento)")
        print("      Tempo estimado: 6-12 horas\n")
    
    try:
        # Usar função train_yolo_model definida acima
        model, results = train_yolo_model(
            config_path=str(config_path),
            model_size='m',
            epochs=10,
            imgsz=640,
            batch_size=batch_size_to_use,
            patience=20,
            device=device_to_use  # Detectado automaticamente
        )
        
        print(f"\n✅ TREINAMENTO CONCLUÍDO COM SUCESSO!")
        print(f"   📁 Resultados salvos em: solar_panel_detection/yolov8m_solar/")
        print(f"   • Melhor modelo: weights/best.pt")
        print(f"   • Última versão: weights/last.pt\n")
        
    except Exception as e:
        print(f"\n❌ ERRO durante o treinamento: {str(e)}")
        if "CUDA" in str(e) or "cuda" in str(e):
            print(f"   💡 Dica: Erro de GPU/CUDA detectado")
            print(f"      Para usar CPU, altere: device='cpu'")
        print("\n")
        raise

print("="*70 + "\n")


🚀 TREINANDO MODELO YOLO V8 - SOLAR PANEL DETECTION

✅ Dataset pronto: 3312 imagens encontradas

1️⃣ Criando arquivo de configuração YAML...
   ✅ Arquivo criado: /home/jovyan/work/solar_panels_data.yaml

2️⃣ Iniciando treinamento YOLOv8m...
   📊 Configuração:
      • Device: CPU
      • Epochs: 100
      • Batch size: 8
      • Tamanho de imagem: 640x640
      • Early stopping: 20 epochs

   ⚠️  GPU não disponível - usando CPU (treinamento será mais lento)
      Tempo estimado: 6-12 horas


🚀 Iniciando treinamento YOLOv8m...
   Device: CPU
   Epochs: 10 | Batch: 8 | Imgsz: 640
Ultralytics 8.4.8 🚀 Python-3.11.6 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-1355U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=solar_panels_data.yaml, degrees=10, determinist

In [8]:
print_section("SEÇÃO 2: DETECÇÃO & AVALIAÇÃO")

# ============================================================================
# 2.1 - Detecção de Painéis Solares
# ============================================================================

def detect_solar_panels(
    model: YOLO,
    image_path: str,
    confidence_threshold: float = 0.5
) -> Dict:
    """
    Detecta painéis solares em uma imagem individual.
    
    Args:
        model: Modelo YOLO treinado
        image_path: Caminho da imagem
        confidence_threshold: Confiança mínima para detecção
    
    Returns:
        Dict com informações de detecção:
            - image_path: Caminho da imagem
            - num_panels_detected: Número de painéis detectados
            - detections: Lista com coordenadas e confiança
            - yolo_result: Objeto original de resultado YOLO
    
    Example:
        >>> result = detect_solar_panels(model, './image.jpg', confidence=0.5)
        >>> print(f"Detectados: {result['num_panels_detected']} painéis")
    """
    results = model.predict(
        source=image_path,
        conf=confidence_threshold,
        verbose=False
    )
    result = results[0]
    
    detections = []
    
    for box in result.boxes:
        coords = box.xyxy[0].cpu().numpy()
        detection = {
            'x_min': float(coords[0]),
            'y_min': float(coords[1]),
            'x_max': float(coords[2]),
            'y_max': float(coords[3]),
            'confidence': float(box.conf[0]),
            'class': int(box.cls[0])
        }
        
        # Calcular dimensões
        detection['width'] = detection['x_max'] - detection['x_min']
        detection['height'] = detection['y_max'] - detection['y_min']
        detection['area_pixels'] = detection['width'] * detection['height']
        
        detections.append(detection)
    
    return {
        'image_path': image_path,
        'num_panels_detected': len(detections),
        'detections': detections,
        'yolo_result': result,
        'timestamp': pd.Timestamp.now()
    }


def process_batch_images(
    model: YOLO,
    image_directory: str,
    output_csv_path: Optional[str] = None,
    confidence_threshold: float = 0.5
) -> Tuple[List[Dict], pd.DataFrame]:
    """
    Processa um diretório completo de imagens.
    
    Args:
        model: Modelo YOLO treinado
        image_directory: Diretório com imagens
        output_csv_path: Caminho para salvar CSV com resumo
        confidence_threshold: Confiança mínima para detecção
    
    Returns:
        Tupla (resultados detalhados, DataFrame resumido)
    """
    image_dir = Path(image_directory)
    image_files = sorted(image_dir.glob('*.jpg')) + sorted(image_dir.glob('*.png'))
    
    if not image_files:
        print(f"⚠️ Nenhuma imagem encontrada em {image_directory}")
        return [], pd.DataFrame()
    
    print(f"🔍 Processando {len(image_files)} imagens...")
    
    all_results = []
    for idx, img_file in enumerate(image_files, 1):
        result = detect_solar_panels(model, str(img_file), confidence_threshold)
        all_results.append(result)
        
        if idx % 10 == 0:
            print(f"   Processadas {idx}/{len(image_files)} imagens...")
    
    # Criar DataFrame com resumo
    summary_data = []
    for result in all_results:
        avg_conf = np.mean([d['confidence'] for d in result['detections']]) if result['detections'] else 0.0
        summary_data.append({
            'image': Path(result['image_path']).name,
            'num_detections': result['num_panels_detected'],
            'avg_confidence': avg_conf,
            'total_area_pixels': sum(d['area_pixels'] for d in result['detections']),
            'processing_time': result.get('timestamp', pd.Timestamp.now())
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    if output_csv_path:
        output_csv_path = Path(output_csv_path)
        output_csv_path.parent.mkdir(parents=True, exist_ok=True)
        summary_df.to_csv(output_csv_path, index=False)
        print(f"✓ Resultados salvos em: {output_csv_path}")
    
    print(f"✓ Processamento completo: {len(all_results)} imagens")
    
    return all_results, summary_df


# ============================================================================
# 2.2 - Avaliação do Modelo
# ============================================================================

def evaluate_model(
    model: YOLO,
    test_config_path: str
) -> Tuple[Dict, any]:
    """
    Avalia modelo YOLO no conjunto de teste.
    
    Args:
        model: Modelo YOLO treinado
        test_config_path: Caminho do arquivo data.yaml
    
    Returns:
        Tupla (dicionário de métricas, objeto de resultado)
    """
    print("\n📊 Avaliando modelo no conjunto de teste...")
    
    results = model.val(data=test_config_path, verbose=False)
    
    # Extrair métricas principais
    metrics_dict = {
        'mAP@50': float(results.box.map50) if hasattr(results.box, 'map50') else 0.0,
        'mAP@50:95': float(results.box.map) if hasattr(results.box, 'map') else 0.0,
        'Precision': float(results.box.mp) if hasattr(results.box, 'mp') else 0.0,
        'Recall': float(results.box.mr) if hasattr(results.box, 'mr') else 0.0,
    }
    
    # Exibir métricas
    print("\n" + "="*60)
    print("MÉTRICAS DE DESEMPENHO")
    print("="*60)
    for metric, value in metrics_dict.items():
        status = "✓" if value > 0.5 else "⚠"
        print(f"{status} {metric:.<40} {value:.4f}")
    print("="*60 + "\n")
    
    return metrics_dict, results


def plot_training_results(results_dir: str) -> Optional[go.Figure]:
    """
    Plota resultados do treinamento a partir do diretório de saída.
    
    Args:
        results_dir: Diretório com logs de treinamento
    
    Returns:
        Figura Plotly (ou None se arquivo não encontrado)
    """
    results_csv = Path(results_dir) / 'results.csv'
    
    if not results_csv.exists():
        print(f"⚠️ Arquivo não encontrado: {results_csv}")
        return None
    
    df = pd.read_csv(results_csv)
    
    # Limpar nomes de colunas (remover espaços extras)
    df.columns = df.columns.str.strip()
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Box Loss', 'Classification Loss',
            'Precision', 'Recall'
        )
    )
    
    # Box Loss
    if 'train/box_loss' in df.columns:
        fig.add_trace(
            go.Scatter(
                y=df['train/box_loss'],
                name='Train Box Loss',
                line=dict(color='red')
            ),
            row=1, col=1
        )
    
    # Classification Loss
    if 'train/cls_loss' in df.columns:
        fig.add_trace(
            go.Scatter(
                y=df['train/cls_loss'],
                name='Train Cls Loss',
                line=dict(color='blue')
            ),
            row=1, col=2
        )
    
    # Precision
    if 'metrics/precision' in df.columns:
        fig.add_trace(
            go.Scatter(
                y=df['metrics/precision'],
                name='Precision',
                line=dict(color='green')
            ),
            row=2, col=1
        )
    
    # Recall
    if 'metrics/recall' in df.columns:
        fig.add_trace(
            go.Scatter(
                y=df['metrics/recall'],
                name='Recall',
                line=dict(color='orange')
            ),
            row=2, col=2
        )
    
    fig.update_layout(
        height=800,
        showlegend=True,
        title_text="📈 Resultados de Treinamento YOLO"
    )
    
    return fig

print("✓ Funções de detecção e avaliação definidas")



  SEÇÃO 2: DETECÇÃO & AVALIAÇÃO

✓ Funções de detecção e avaliação definidas


## 6. Classify Property Type

Classificação baseada em características dos painéis detectados:
- **Residencial**: 1-5 painéis pequenos (3-10 kW)
- **Comercial**: 5-20 painéis médios (10-50 kW)
- **Industrial**: 20+ painéis grandes (50+ kW)
- **Subestação**: Arranjo muito estruturado, alta densidade

In [9]:
class PropertyClassifier:
    """
    Classificador de tipo de propriedade baseado em detecção de painéis solares
    """
    
    def __init__(self):
        self.property_types = {
            'residential': {'min_panels': 1, 'max_panels': 5, 'power_range': (3, 10)},
            'commercial': {'min_panels': 5, 'max_panels': 20, 'power_range': (10, 50)},
            'industrial': {'min_panels': 20, 'max_panels': 500, 'power_range': (50, 500)},
            'substation': {'min_panels': 500, 'max_panels': float('inf'), 'power_range': (500, 10000)}
        }
    
    def extract_features(self, detections):
        """
        Extrai características dos painéis detectados
        """
        if not detections:
            return None
        
        areas = [d['area_pixels'] for d in detections]
        confidences = [d['confidence'] for d in detections]
        
        features = {
            'num_panels': len(detections),
            'avg_area': np.mean(areas),
            'std_area': np.std(areas),
            'total_area': np.sum(areas),
            'avg_confidence': np.mean(confidences),
            'min_confidence': np.min(confidences),
            'max_confidence': np.max(confidences),
            'area_variance': np.var(areas),  # Variância (subestações têm painéis muito uniformes)
            'coverage_ratio': np.sum(areas) / (1920 * 1080)  # Assumindo 1920x1080
        }
        
        return features
    
    def classify(self, detections, estimated_power=None):
        """
        Classifica o tipo de propriedade
        
        Args:
            detections: lista de detecções YOLO
            estimated_power: potência estimada em kW (opcional)
        
        Returns:
            classificação, confiança, features
        """
        if not detections or len(detections) == 0:
            return 'unknown', 0.0, None
        
        features = self.extract_features(detections)
        num_panels = features['num_panels']
        area_variance = features['area_variance']
        
        # Lógica de classificação
        if num_panels < 5:
            if area_variance < 0.1:  # Painéis muito uniformes
                return 'residential', 0.9, features
            else:
                return 'residential', 0.8, features
        
        elif 5 <= num_panels < 20:
            if area_variance < 0.05:  # Muito uniforme
                return 'commercial', 0.85, features
            else:
                return 'commercial', 0.75, features
        
        elif 20 <= num_panels < 100:
            if area_variance < 0.03:  # Altamente uniforme (subestação)
                return 'substation', 0.88, features
            else:
                return 'industrial', 0.8, features
        
        else:  # > 100 painéis
            if area_variance < 0.02:
                return 'substation', 0.95, features
            else:
                return 'industrial', 0.9, features
    
    def generate_report(self, image_path, detections, classification):
        """
        Gera relatório de classificação
        """
        prop_class, confidence, features = classification
        
        report = {
            'image': Path(image_path).name,
            'property_type': prop_class,
            'classification_confidence': confidence,
            'num_panels': len(detections) if detections else 0,
            'features': features
        }
        
        return report

# Inicializar classificador
classifier = PropertyClassifier()
print("✓ Classificador de propriedades criado")

✓ Classificador de propriedades criado


## 7. Estimate Solar Panel Power Output

Estimativa de potência baseada em:
- Área detectada em pixels
- Conversão pixel → metros reais
- Especificações padrão de painéis (150-400 W/m²)

In [10]:
class PowerEstimator:
    """
    Estimador de potência de painéis solares
    """
    
    # Padrões de painéis solares
    PANEL_SPECS = {
        'standard_6inch': {'size_m2': 1.65, 'power_w': 300},  # Painel 6x3 pés
        'standard_5inch': {'size_m2': 2.0, 'power_w': 400},   # Painel padrão
        'mini': {'size_m2': 0.5, 'power_w': 100},
        'large': {'size_m2': 2.5, 'power_w': 500}
    }
    
    def __init__(self, camera_specs=None):
        """
        Inicializa estimador
        
        Args:
            camera_specs: dict com informações da câmera
                - focal_length: distância focal em mm
                - sensor_width: largura do sensor em mm
                - image_width: largura da imagem em pixels
                - altitude: altitude de captura em metros (para drones)
        """
        self.camera_specs = camera_specs or {
            'focal_length': 50,
            'sensor_width': 36,
            'image_width': 1920,
            'altitude': 100  # metros
        }
    
    def pixels_to_meters(self, pixel_area, image_resolution=(1920, 1080)):
        """
        Converte área em pixels para metros quadrados
        
        Usa o tamanho típico de um painel solar (2m x 1m = 2m²) como referência
        """
        # Proporção de área típica de um painel
        typical_panel_pixels = (100 * 50)  # 100x50 pixels aproximadamente
        typical_panel_area_m2 = 2.0
        
        # Converter
        area_m2 = (pixel_area / typical_panel_pixels) * typical_panel_area_m2
        return area_m2
    
    def estimate_power(self, detections, efficiency=0.15, power_density=150):
        """
        Estima potência total dos painéis
        
        Args:
            detections: lista de detecções YOLO
            efficiency: eficiência do painel (0.15 = 15%)
            power_density: W/m² (150-200 para painéis médios)
        
        Returns:
            dict com estimativas
        """
        if not detections:
            return {'total_power_kw': 0, 'total_area_m2': 0, 'num_panels': 0}
        
        total_pixels = sum(d['area_pixels'] for d in detections)
        total_area_m2 = self.pixels_to_meters(total_pixels)
        
        # Potência = Área * Densidade * Eficiência
        total_power_w = total_area_m2 * power_density * efficiency
        total_power_kw = total_power_w / 1000
        
        # Potência por painel (assumindo painel médio de 400W)
        avg_power_per_panel = 0.4  # kW
        estimated_num_panels = len(detections)
        
        return {
            'total_power_kw': total_power_kw,
            'power_estimate_method': 'area_based',
            'total_area_m2': total_area_m2,
            'num_panels_detected': len(detections),
            'avg_power_per_panel_kw': avg_power_per_panel,
            'power_from_count_kw': estimated_num_panels * avg_power_per_panel,
            'power_density_used': power_density,
            'efficiency_used': efficiency
        }
    
    def estimate_annual_production(self, power_kw, location='Brazil', capacity_factor=0.18):
        """
        Estima produção anual de energia
        
        Args:
            power_kw: potência instalada em kW
            location: localização geográfica
            capacity_factor: fator de capacidade (0.18 = 18% para Brasil)
        
        Returns:
            dict com estimativas de produção
        """
        # Horas por ano
        hours_per_year = 365.25 * 24
        
        # Produção (kWh/ano)
        annual_production_kwh = power_kw * hours_per_year * capacity_factor
        
        # Economia estimada (R$/kWh médio = R$ 0.80)
        tariff = 0.80
        annual_savings = annual_production_kwh * tariff
        
        return {
            'annual_production_kwh': annual_production_kwh,
            'daily_avg_kwh': annual_production_kwh / 365.25,
            'annual_savings_brl': annual_savings,
            'capacity_factor': capacity_factor,
            'location': location
        }

# Inicializar estimador
estimator = PowerEstimator()
print("✓ Estimador de potência criado")

✓ Estimador de potência criado


In [11]:
print_section("SEÇÃO 3: VISUALIZAÇÃO & RELATÓRIOS")

# ============================================================================
# 3.1 - Visualização de Detecções
# ============================================================================

def visualize_detections(
    image_path: str,
    detections: List[Dict],
    title: str = "Detecção de Painéis Solares",
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (15, 10)
) -> None:
    """
    Visualiza detecções YOLO com bounding boxes sobre a imagem.
    
    Args:
        image_path: Caminho da imagem
        detections: Lista de detecções (dicts com coordenadas e confiança)
        title: Título da figura
        save_path: Caminho para salvar a figura (opcional)
        figsize: Tamanho da figura (largura, altura)
    
    Example:
        >>> visualize_detections('./image.jpg', detections, save_path='./output.png')
    """
    import cv2
    
    # Carregar imagem
    img = cv2.imread(image_path)
    if img is None:
        print(f"⚠️ Erro ao carregar imagem: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Desenhar bounding boxes
    for detection in detections:
        x_min = int(detection['x_min'])
        y_min = int(detection['y_min'])
        x_max = int(detection['x_max'])
        y_max = int(detection['y_max'])
        conf = detection['confidence']
        
        # Cor baseada em confiança (verde = alta, vermelho = baixa)
        color = (
            int(255 * (1 - conf)),  # R: aumenta com baixa confiança
            int(255 * conf),        # G: aumenta com alta confiança
            0                        # B: neutro
        )
        
        # Desenhar retângulo
        cv2.rectangle(img_rgb, (x_min, y_min), (x_max, y_max), color, 2)
        
        # Desenhar label com confiança
        label = f'{conf:.2f}'
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(img_rgb, label, (x_min, y_min - 10), font, 0.6, color, 2)
    
    # Plotar
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'{title} - {len(detections)} painéis detectados')
    
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Imagem salva em: {save_path}")
    
    plt.show()


def visualize_batch_results(
    results_df: pd.DataFrame,
    save_path: Optional[str] = None
) -> go.Figure:
    """
    Visualiza resumo de resultados em batch (múltiplas imagens).
    
    Args:
        results_df: DataFrame com colunas: image, num_detections, avg_confidence, total_area_pixels
        save_path: Caminho para salvar figura HTML
    
    Returns:
        Figura Plotly interativa
    """
    if results_df.empty:
        print("⚠️ DataFrame vazio!")
        return None
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"type": "bar"}, {"type": "scatter"}],
            [{"type": "histogram"}, {"type": "scatter"}]
        ],
        subplot_titles=(
            'Detecções por Imagem',
            'Confiança vs Área',
            'Distribuição de Detecções',
            'Área Total por Imagem'
        )
    )
    
    # 1. Gráfico de barras: número de detecções
    fig.add_trace(
        go.Bar(
            x=results_df['image'],
            y=results_df['num_detections'],
            name='Num. Detecções',
            marker_color='lightblue'
        ),
        row=1, col=1
    )
    
    # 2. Scatter: Confiança vs Área
    fig.add_trace(
        go.Scatter(
            x=results_df['total_area_pixels'],
            y=results_df['avg_confidence'],
            mode='markers',
            name='Detecções',
            marker=dict(
                size=results_df['num_detections'] * 2,
                color=results_df['avg_confidence'],
                colorscale='Viridis',
                showscale=True
            ),
            text=results_df['image'],
            hovertemplate='<b>%{text}</b><br>Área: %{x}<br>Confiança: %{y:.2f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # 3. Histograma de detecções
    fig.add_trace(
        go.Histogram(
            x=results_df['num_detections'],
            name='Distribuição',
            nbinsx=20,
            marker_color='rgba(100, 200, 255, 0.7)'
        ),
        row=2, col=1
    )
    
    # 4. Scatter: Área total
    fig.add_trace(
        go.Scatter(
            x=results_df['image'],
            y=results_df['total_area_pixels'],
            mode='markers+lines',
            name='Área Total',
            marker=dict(size=8, color='darkblue'),
            line=dict(color='darkblue', width=1)
        ),
        row=2, col=2
    )
    
    fig.update_layout(
        height=900,
        showlegend=True,
        title_text="📊 Análise em Batch - Detecção de Painéis Solares",
        hovermode='closest'
    )
    
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(save_path)
        print(f"✓ Gráfico salvo em: {save_path}")
    
    return fig


# ============================================================================
# 3.2 - Relatório Completo
# ============================================================================

def create_analysis_report(
    image_path: str,
    detections: List[Dict],
    classification: Tuple[str, float, Dict],
    power_estimate: Dict,
    save_path: Optional[str] = None
) -> go.Figure:
    """
    Cria relatório completo e interativo de análise.
    
    Args:
        image_path: Caminho da imagem processada
        detections: Lista de detecções
        classification: Tupla (type, confidence, features)
        power_estimate: Dict com estimativas de potência
        save_path: Caminho para salvar relatório HTML
    
    Returns:
        Figura Plotly com 4 subgráficos
    """
    prop_class, conf, features = classification
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"type": "pie"}, {"type": "bar"}],
            [{"type": "scatter"}, {"type": "indicator"}]
        ],
        subplot_titles=(
            'Distribuição de Confiança',
            'Estatísticas de Área',
            'Área vs Confiança por Painel',
            'Potência Estimada'
        )
    )
    
    # 1. Pie chart: distribuição de confiança
    confidences = [d['confidence'] for d in detections]
    high_conf = sum(1 for c in confidences if c > 0.8)
    med_conf = sum(1 for c in confidences if 0.6 <= c <= 0.8)
    low_conf = sum(1 for c in confidences if c < 0.6)
    
    fig.add_trace(
        go.Pie(
            labels=['🟢 Alta (>0.8)', '🟡 Média (0.6-0.8)', '🔴 Baixa (<0.6)'],
            values=[high_conf, med_conf, low_conf],
            name='Confiança',
            marker=dict(colors=['green', 'orange', 'red'])
        ),
        row=1, col=1
    )
    
    # 2. Bar chart: estatísticas de área
    areas = [d['area_pixels'] for d in detections]
    if areas:
        fig.add_trace(
            go.Bar(
                x=['Min', 'Média', 'Max', 'Total'],
                y=[min(areas), np.mean(areas), max(areas), sum(areas)],
                name='Área (pixels)',
                marker_color=['lightcoral', 'lightskyblue', 'lightgreen', 'lightseagreen']
            ),
            row=1, col=2
        )
    
    # 3. Scatter: Área vs Confiança
    if areas and confidences:
        fig.add_trace(
            go.Scatter(
                x=areas,
                y=confidences,
                mode='markers',
                name='Painéis',
                marker=dict(
                    size=8,
                    color=confidences,
                    colorscale='Viridis',
                    showscale=True,
                    line=dict(width=1, color='white')
                ),
                text=[f"Painel {i+1}" for i in range(len(areas))],
                hovertemplate='<b>%{text}</b><br>Área: %{x} px<br>Confiança: %{y:.2%}<extra></extra>'
            ),
            row=2, col=1
        )
    
    # 4. Indicador: Potência Estimada
    power_kw = power_estimate.get('total_power_kw', 0)
    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=power_kw,
            title={'text': "Potência (kW)"},
            domain={'x': [0, 1], 'y': [0, 1]},
            number={'suffix': " kW", 'font': {'size': 40, 'color': 'darkblue'}}
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_xaxes(title_text="Área (pixels)", row=2, col=1)
    fig.update_yaxes(title_text="Confiança", row=2, col=1)
    
    fig.update_layout(
        height=900,
        showlegend=True,
        title_text=f"📋 Relatório Completo: {Path(image_path).name}",
        font=dict(size=11)
    )
    
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(save_path)
        print(f"✓ Relatório salvo em: {save_path}")
    
    return fig


print("✓ Funções de visualização e relatórios definidas")



  SEÇÃO 3: VISUALIZAÇÃO & RELATÓRIOS

✓ Funções de visualização e relatórios definidas


In [12]:
print_section("SEÇÃO 4: PIPELINE COMPLETO")

# ============================================================================
# 4.1 - Pipeline de Processamento Integrado
# ============================================================================

def full_pipeline(
    image_path: str,
    model: YOLO,
    classifier: 'PropertyClassifier',
    estimator: 'PowerEstimator',
    confidence_threshold: float = 0.5,
    save_results: bool = False,
    output_dir: Optional[str] = None
) -> Dict:
    """
    Pipeline COMPLETO de análise: Detecção → Classificação → Estimativa de Potência.
    
    Processa uma imagem em 4 etapas:
    1. Detecção de painéis solares com YOLOv8
    2. Classificação de tipo de propriedade
    3. Estimativa de potência instalada
    4. Cálculo de produção e economia anual
    
    Args:
        image_path: Caminho da imagem para análise
        model: Modelo YOLO treinado (instância de YOLO)
        classifier: Classificador de propriedade (instância de PropertyClassifier)
        estimator: Estimador de potência (instância de PowerEstimator)
        confidence_threshold: Confiança mínima para detecção (0.0-1.0)
        save_results: Se True, salva visualizações e relatórios
        output_dir: Diretório para salvar outputs (se save_results=True)
    
    Returns:
        Dict contendo:
            - image_path: Caminho da imagem processada
            - num_detections: Número de painéis detectados
            - detections: Lista detalhada de detecções
            - classification: Tipo, confiança e features
            - power_estimate: Potência e estatísticas
            - annual_production: Produção anual e economia
            - timestamps: Tempos de processamento
    
    Example:
        >>> model = YOLO('modelos/yolo_solar_panel.pt')
        >>> results = full_pipeline(
        ...     'imagem.jpg',
        ...     model,
        ...     classifier,
        ...     estimator,
        ...     save_results=True,
        ...     output_dir='./outputs'
        ... )
        >>> print(f"Potência: {results['power_estimate']['total_power_kw']:.2f} kW")
    """
    import time
    
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Inicializar timestamps
    timestamps = {'inicio': time.time()}
    
    print(f"\n{'='*70}")
    print(f"  PROCESSANDO: {Path(image_path).name}")
    print(f"{'='*70}\n")
    
    # ────────────────────────────────────────────────────────────────────────
    # ETAPA 1: DETECÇÃO
    # ────────────────────────────────────────────────────────────────────────
    
    timestamps['deteccao_inicio'] = time.time()
    print("▶️  ETAPA 1/4: Detectando painéis solares...")
    
    detection_results = detect_solar_panels(model, image_path, confidence_threshold)
    detections = detection_results['detections']
    
    print(f"✓ {len(detections)} painéis detectados")
    if detections:
        avg_conf = np.mean([d['confidence'] for d in detections])
        print(f"  • Confiança média: {avg_conf:.2%}")
        print(f"  • Área total: {sum(d['area_pixels'] for d in detections):.0f} pixels")
    
    timestamps['deteccao_fim'] = time.time()
    
    # ────────────────────────────────────────────────────────────────────────
    # ETAPA 2: CLASSIFICAÇÃO
    # ────────────────────────────────────────────────────────────────────────
    
    timestamps['classificacao_inicio'] = time.time()
    print("\n▶️  ETAPA 2/4: Classificando tipo de propriedade...")
    
    classification = classifier.classify(detections)
    prop_type, conf, features = classification
    
    print(f"✓ Tipo identificado: {prop_type.upper()}")
    print(f"  • Confiança: {conf:.2%}")
    if features:
        print(f"  • Cobertura: {features['coverage_ratio']:.2%}")
    
    timestamps['classificacao_fim'] = time.time()
    
    # ────────────────────────────────────────────────────────────────────────
    # ETAPA 3: ESTIMATIVA DE POTÊNCIA
    # ────────────────────────────────────────────────────────────────────────
    
    timestamps['potencia_inicio'] = time.time()
    print("\n▶️  ETAPA 3/4: Estimando potência instalada...")
    
    power_estimate = estimator.estimate_power(detections)
    
    print(f"✓ Potência estimada: {power_estimate['total_power_kw']:.2f} kW")
    print(f"  • Método: {power_estimate['power_estimate_method']}")
    print(f"  • Área: {power_estimate['total_area_m2']:.2f} m²")
    print(f"  • Painéis estimados: {power_estimate['num_panels_detected']}")
    
    timestamps['potencia_fim'] = time.time()
    
    # ────────────────────────────────────────────────────────────────────────
    # ETAPA 4: PRODUÇÃO ANUAL & ECONOMIA
    # ────────────────────────────────────────────────────────────────────────
    
    timestamps['producao_inicio'] = time.time()
    print("\n▶️  ETAPA 4/4: Calculando produção anual...")
    
    annual_production = estimator.estimate_annual_production(
        power_estimate['total_power_kw']
    )
    
    print(f"✓ Produção estimada: {annual_production['annual_production_kwh']:.0f} kWh/ano")
    print(f"  • Produção diária: {annual_production['daily_avg_kwh']:.1f} kWh/dia")
    print(f"  • Economia anual: R$ {annual_production['annual_savings_brl']:,.2f}")
    print(f"  • Fator de capacidade: {annual_production['capacity_factor']:.1%}")
    
    timestamps['producao_fim'] = time.time()
    
    # ────────────────────────────────────────────────────────────────────────
    # GERAR OUTPUTS (se solicitado)
    # ────────────────────────────────────────────────────────────────────────
    
    if save_results and output_dir:
        print("\n▶️  Gerando visualizações...")
        
        # Visualizar detecções
        vis_path = output_dir / f"deteccoes_{Path(image_path).stem}.png"
        visualize_detections(image_path, detections, save_path=str(vis_path))
        
        # Relatório interativo
        report_path = output_dir / f"relatorio_{Path(image_path).stem}.html"
        create_analysis_report(
            image_path, detections, classification, power_estimate,
            save_path=str(report_path)
        )
    
    # ────────────────────────────────────────────────────────────────────────
    # COMPILAR RESULTADOS
    # ────────────────────────────────────────────────────────────────────────
    
    timestamps['fim'] = time.time()
    
    results = {
        'image_path': image_path,
        'image_name': Path(image_path).name,
        'num_detections': len(detections),
        'detections': detections,
        'classification': {
            'type': prop_type,
            'confidence': conf,
            'features': features
        },
        'power_estimate': power_estimate,
        'annual_production': annual_production,
        'yolo_result': detection_results.get('yolo_result'),
        'timestamps': {
            'deteccao': timestamps['deteccao_fim'] - timestamps['deteccao_inicio'],
            'classificacao': timestamps['classificacao_fim'] - timestamps['classificacao_inicio'],
            'potencia': timestamps['potencia_fim'] - timestamps['potencia_inicio'],
            'producao': timestamps['producao_fim'] - timestamps['producao_inicio'],
            'total': timestamps['fim'] - timestamps['inicio']
        }
    }
    
    # Resumo final
    print("\n" + "="*70)
    print(f"✅ ANÁLISE CONCLUÍDA EM {results['timestamps']['total']:.2f}s")
    print("="*70)
    print(f"""
    📊 RESUMO EXECUTIVO:
    ├─ Painéis Detectados: {len(detections)}
    ├─ Tipo de Propriedade: {prop_type.upper()}
    ├─ Potência Instalada: {power_estimate['total_power_kw']:.2f} kW
    ├─ Produção Anual: {annual_production['annual_production_kwh']:.0f} kWh
    └─ Economia Anual: R$ {annual_production['annual_savings_brl']:,.2f}
    """)
    
    return results


# Exemplo de uso (descomente quando tiver modelo e dados):
# >>> model = YOLO('modelos/yolo_solar_panel.pt')
# >>> results = full_pipeline(
# ...     'sample_image.jpg',
# ...     model,
# ...     classifier,
# ...     estimator,
# ...     save_results=True,
# ...     output_dir='./outputs'
# ... )

print("✓ Pipeline completo definido - Pronto para uso!")



  SEÇÃO 4: PIPELINE COMPLETO

✓ Pipeline completo definido - Pronto para uso!


In [13]:
def process_directory(directory, model, classifier, estimator, output_dir=None):
    """
    Processa todas as imagens em um diretório
    
    Args:
        directory: diretório com imagens
        model: modelo YOLO
        classifier: classificador
        estimator: estimador
        output_dir: diretório para salvar resultados
    
    Returns:
        DataFrame com resultados
    """
    
    if output_dir is None:
        output_dir = Path(directory) / 'results'
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    image_dir = Path(directory)
    image_files = sorted(image_dir.glob('*.jpg')) + sorted(image_dir.glob('*.png'))
    
    results_list = []
    
    for idx, img_file in enumerate(image_files, 1):
        print(f"\n[{idx}/{len(image_files)}] Processando: {img_file.name}")
        
        try:
            result = full_pipeline(str(img_file), model, classifier, estimator)
            
            # Extrair dados para DataFrame
            result_row = {
                'image_name': img_file.name,
                'num_panels': result['num_detections'],
                'property_type': result['classification']['type'],
                'classification_confidence': result['classification']['confidence'],
                'power_kw': result['power_estimate']['total_power_kw'],
                'area_m2': result['power_estimate']['total_area_m2'],
                'annual_production_kwh': result['annual_production']['annual_production_kwh'],
                'annual_savings_brl': result['annual_production']['annual_savings_brl']
            }
            
            results_list.append(result_row)
            
            # Visualizar
            visualize_detections(
                str(img_file),
                result['detections'],
                save_path=str(output_dir / f"{img_file.stem}_detections.png")
            )
            
        except Exception as e:
            print(f"❌ Erro ao processar {img_file.name}: {str(e)}")
            continue
    
    # Criar DataFrame e salvar
    results_df = pd.DataFrame(results_list)
    
    csv_path = output_dir / 'results_summary.csv'
    results_df.to_csv(csv_path, index=False)
    print(f"\n✓ Resultados salvos em: {csv_path}")
    
    # Estatísticas gerais
    print("\n" + "="*60)
    print("RESUMO GERAL")
    print("="*60)
    print(results_df.describe())
    print("="*60)
    
    return results_df, output_dir

def export_results_to_json(results_dict, output_path):
    """
    Exporta resultados para JSON
    """
    import json
    
    # Converter arrays numpy para listas
    def convert_to_serializable(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.generic):
            return obj.item()
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(item) for item in obj]
        return obj
    
    serializable_results = convert_to_serializable(results_dict)
    
    with open(output_path, 'w') as f:
        json.dump(serializable_results, f, indent=2)
    
    print(f"✓ Resultados exportados para: {output_path}")

print("✓ Funções de batch processing e export criadas")

✓ Funções de batch processing e export criadas


## 📖 Guia de Integração com Dados Existentes

### Carregar Dataset Lacuna Solar Survey

```python
# Importar dados do CSV
survey_df = pd.read_csv('./data/lacuna-solar-survey-zindi/test.csv')
print(f"Total de imagens: {len(survey_df)}")

# Processar em batch
image_dir = './data/lacuna-solar-survey-zindi/images'
detections_list, summary_df = process_batch_images(
    model,
    image_dir,
    output_csv_path='./output/deteccoes_survey.csv'
)

# Salvar com GIS se disponível
if 'lat' in survey_df.columns and 'lon' in survey_df.columns:
    summary_df = summary_df.merge(
        survey_df[['image_id', 'lat', 'lon']],
        left_on='image', right_on='image_id'
    )
    # Opcional: Exportar para GeoJSON para visualização em mapa
```

### Integração com Dados de Subestações

```python
# Carregar dados de subestações
substacoes_df = pd.read_csv('./data/subestacoes_data.csv')

# Enriquecer com detecções YOLO
for idx, row in substacoes_df.iterrows():
    image_path = f"./data/substacao_imagens/{row['substacao_id']}.jpg"
    if Path(image_path).exists():
        result = full_pipeline(
            image_path,
            model,
            classifier,
            estimator
        )
        substacoes_df.loc[idx, 'panels_detected'] = result['num_detections']
        substacoes_df.loc[idx, 'power_kw'] = result['power_estimate']['total_power_kw']
        substacoes_df.loc[idx, 'property_type'] = result['classification']['type']
```

### Exportar Resultados

```python
# CSV simples
summary_df.to_csv('./output/deteccoes_resumo.csv', index=False)

# JSON com detalhes
import json
with open('./output/deteccoes_completo.json', 'w') as f:
    json.dump([r.__dict__ for r in detections_list], f, indent=2)

# GeoJSON (se tiver coordenadas)
from geojson import Feature, FeatureCollection
features = [
    Feature(
        geometry={"type": "Point", "coordinates": [row['lon'], row['lat']]},
        properties=row.to_dict()
    )
    for _, row in summary_df.iterrows()
]
with open('./output/mapa_painel.geojson', 'w') as f:
    json.dump(FeatureCollection(features), f, indent=2)
```

## APÊNDICE C: Integração com Dataset Existente

Use seus dados e imagens já coletadas com este guia

In [14]:
# Carregar dados CSV existentes
import os

# Caminho do seu arquivo CSV (agora no local correto)
csv_file_path = 'data/lacuna-solar-survey-zindi-2/test.csv'
images_dir = 'data/lacuna-solar-survey-zindi-2/images'

# Verificar se o arquivo existe
if os.path.exists(csv_file_path):
    metadata_df = pd.read_csv(csv_file_path)
    print(f"✓ CSV carregado com sucesso!")
    print(f"  📁 Localização: {os.path.abspath(csv_file_path)}")
    print(f"  📊 Linhas: {len(metadata_df)}")
    print(f"  📋 Colunas: {list(metadata_df.columns)}\n")
    print("Primeiras linhas:")
    print(metadata_df.head(10))
    
    # Verificar se as imagens existem
    if os.path.exists(images_dir):
        num_images = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.png'))])
        print(f"\n✓ Diretório de imagens encontrado!")
        print(f"  📸 Total de imagens: {num_images}")
    else:
        print(f"\n❌ Diretório de imagens não encontrado: {images_dir}")
else:
    print(f"❌ Arquivo não encontrado: {csv_file_path}")
    metadata_df = None

✓ CSV carregado com sucesso!
  📁 Localização: /home/jovyan/work/data/lacuna-solar-survey-zindi-2/test.csv
  📊 Linhas: 1107
  📋 Colunas: ['ID', 'img_origin', 'placement']

Primeiras linhas:
                 ID img_origin    placement
0          ID00qprY          D         roof
1         ID01AciUc          D         roof
2           ID0328D          D         roof
3     ID05WxObCFTs9          D         roof
4     ID06AdCmLMlkO          S    S-unknown
5         ID0BTVyh4          D         roof
6    ID0FBDllOkNS3A          D         roof
7    ID0FjL0VeJuPBW          S    S-unknown
8  ID0HlRCFE5wCBkWP          D         roof
9         ID0IovKJW          D  r_openspace

✓ Diretório de imagens encontrado!
  📸 Total de imagens: 4419


In [15]:
def analyze_existing_dataset(metadata_df):
    """
    Analisa o dataset existente e mapeia as informações
    """
    if metadata_df is None:
        print("❌ Nenhum dataset carregado")
        return
    
    print("\n" + "="*70)
    print("📊 ANÁLISE DO DATASET EXISTENTE")
    print("="*70 + "\n")
    
    # Análise por origem
    print("📍 Origem das imagens (img_origin):")
    print(metadata_df['img_origin'].value_counts())
    
    # Análise por placement (tipo de localização)
    print("\n📍 Localização dos painéis (placement):")
    placement_counts = metadata_df['placement'].value_counts()
    print(placement_counts)
    
    # Mapeamento para classes YOLO
    placement_mapping = {
        'roof': 'roof',                    # Telhado - Residencial/Comercial
        'openspace': 'ground_mount',       # Campo aberto - Subestação/Industrial
        'r_openspace': 'ground_mount',     # Rural openspace
        'S-unknown': 'unknown'             # Desconhecido
    }
    
    print("\n🔄 Mapeamento para Classes YOLO:")
    print("""
    'roof'       → Telhado (residencial/comercial)
    'openspace'  → Montagem em solo (industrial/subestação)
    'r_openspace' → Montagem rural em solo
    'S-unknown'  → Desconhecido (será filtrado)
    """)
    
    print("\n✓ Distribuição por classe:")
    metadata_df['yolo_class'] = metadata_df['placement'].map(placement_mapping)
    print(metadata_df['yolo_class'].value_counts())
    
    print("\n" + "="*70)
    
    return placement_mapping

# Executar análise
if metadata_df is not None:
    placement_mapping = analyze_existing_dataset(metadata_df)


📊 ANÁLISE DO DATASET EXISTENTE

📍 Origem das imagens (img_origin):
img_origin
D    913
S    194
Name: count, dtype: int64

📍 Localização dos painéis (placement):
placement
roof           833
S-unknown      194
r_openspace     40
openspace       40
Name: count, dtype: int64

🔄 Mapeamento para Classes YOLO:

    'roof'       → Telhado (residencial/comercial)
    'openspace'  → Montagem em solo (industrial/subestação)
    'r_openspace' → Montagem rural em solo
    'S-unknown'  → Desconhecido (será filtrado)
    

✓ Distribuição por classe:
yolo_class
roof            833
unknown         194
ground_mount     80
Name: count, dtype: int64



In [16]:
def create_yolo_labels_from_metadata(metadata_df, images_dir, output_labels_dir, 
                                     placement_to_class=None):
    """
    Cria labels YOLO a partir dos metadados
    
    Gera bounding boxes automáticos baseado em análise de imagem
    (detecção de regiões com painéis solares)
    
    Args:
        metadata_df: DataFrame com ID, img_origin, placement
        images_dir: diretório com as imagens
        output_labels_dir: diretório para salvar labels
        placement_to_class: mapeamento de placement para class_id YOLO
    """
    if placement_to_class is None:
        placement_to_class = {
            'roof': 0,
            'openspace': 0,
            'r_openspace': 0,
            'S-unknown': -1  # Será ignorado
        }
    
    images_dir = Path(images_dir)
    output_labels_dir = Path(output_labels_dir)
    output_labels_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n📝 Gerando labels YOLO...\n")
    
    created_count = 0
    missing_images = []
    
    for idx, row in metadata_df.iterrows():
        img_id = row['ID']
        placement = row['placement']
        
        # Pular desconhecidos
        if placement == 'S-unknown':
            continue
        
        class_id = placement_to_class.get(placement, 0)
        
        # Procurar imagem
        img_file = None
        for ext in ['.jpg', '.png', '.JPG', '.PNG']:
            potential = images_dir / f"{img_id}{ext}"
            if potential.exists():
                img_file = potential
                break
        
        if img_file is None:
            missing_images.append(img_id)
            continue
        
        # Analisar imagem para encontrar painéis solares
        try:
            img = cv2.imread(str(img_file))
            if img is None:
                continue
            
            height, width = img.shape[:2]
            
            # Estratégia simples: detectar regiões azuis (painéis solares)
            # Converter para HSV para melhor detecção de cores
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            
            # Range para azul
            lower_blue = np.array([100, 50, 50])
            upper_blue = np.array([130, 255, 255])
            mask = cv2.inRange(hsv, lower_blue, upper_blue)
            
            # Se houver painéis detectados
            if np.sum(mask) > (width * height * 0.01):  # Pelo menos 1% da imagem
                # Encontrar contornos
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                annotations = []
                for contour in contours:
                    if cv2.contourArea(contour) < 100:  # Filtrar pequenos ruídos
                        continue
                    
                    x, y, w, h = cv2.boundingRect(contour)
                    
                    # Normalizar para YOLO (0-1)
                    x_center = (x + w/2) / width
                    y_center = (y + h/2) / height
                    norm_width = w / width
                    norm_height = h / height
                    
                    annotations.append(f"{class_id} {x_center:.4f} {y_center:.4f} {norm_width:.4f} {norm_height:.4f}")
                
                # Salvar labels
                if annotations:
                    label_file = output_labels_dir / f"{img_id}.txt"
                    with open(label_file, 'w') as f:
                        f.write('\n'.join(annotations))
                    created_count += 1
                else:
                    # Se não detectou painéis, criar bbox padrão (todo o painel)
                    label_file = output_labels_dir / f"{img_id}.txt"
                    with open(label_file, 'w') as f:
                        f.write(f"{class_id} 0.5 0.5 0.8 0.8")
                    created_count += 1
            else:
                # Sem painéis detectados, criar bbox padrão
                label_file = output_labels_dir / f"{img_id}.txt"
                with open(label_file, 'w') as f:
                    f.write(f"{class_id} 0.5 0.5 0.8 0.8")
                created_count += 1
                
        except Exception as e:
            print(f"  ⚠️  Erro ao processar {img_id}: {str(e)}")
            continue
    
    print(f"✅ Labels criados: {created_count}")
    if missing_images:
        print(f"⚠️  Imagens não encontradas: {len(missing_images)}")
    
    return output_labels_dir

# Exemplo de uso (descomente quando tiver as imagens):
# labels_dir = create_yolo_labels_from_metadata(
#     metadata_df,
#     images_dir='caminho/para/suas/imagens',
#     output_labels_dir=DATASET_PATH / 'temp_labels'
# )

print("✓ Função de conversão de metadados para YOLO criada")

✓ Função de conversão de metadados para YOLO criada


In [17]:
def integrate_existing_data(metadata_df, images_dir, dataset_path, 
                           train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Integra dados existentes no formato YOLO
    
    Pipeline completo:
    1. Gera labels YOLO a partir dos metadados
    2. Copia imagens e labels para estrutura YOLO
    3. Split em treino/validação/teste
    
    Args:
        metadata_df: DataFrame com metadados
        images_dir: diretório com todas as imagens
        dataset_path: diretório de destino
        train_ratio, val_ratio, test_ratio: proporções de split
    """
    import shutil
    
    images_dir = Path(images_dir)
    dataset_path = Path(dataset_path)
    temp_labels_dir = dataset_path / 'temp_labels'
    
    print("\n" + "="*70)
    print("🚀 INTEGRANDO DATASET EXISTENTE")
    print("="*70 + "\n")
    
    # Passo 1: Gerar labels
    print("1️⃣ Gerando labels YOLO a partir dos metadados...")
    labels_dir = create_yolo_labels_from_metadata(
        metadata_df,
        images_dir,
        temp_labels_dir
    )
    
    # Passo 2: Organizar em estrutura YOLO
    print("\n2️⃣ Organizando em estrutura YOLO...")
    
    # Listar imagens que têm labels
    valid_images = []
    for label_file in temp_labels_dir.glob('*.txt'):
        img_id = label_file.stem
        for ext in ['.jpg', '.png', '.JPG', '.PNG']:
            img_file = images_dir / f"{img_id}{ext}"
            if img_file.exists():
                valid_images.append((img_file, label_file))
                break
    
    print(f"   ✓ {len(valid_images)} imagens com labels encontradas")
    
    # Split dataset
    from sklearn.model_selection import train_test_split
    
    indices = list(range(len(valid_images)))
    train_idx, temp_idx = train_test_split(
        indices, test_size=(1-train_ratio), random_state=42
    )
    val_ratio_adj = val_ratio / (val_ratio + test_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=(1-val_ratio_adj), random_state=42
    )
    
    splits = {
        'train': [valid_images[i] for i in train_idx],
        'val': [valid_images[i] for i in val_idx],
        'test': [valid_images[i] for i in test_idx]
    }
    
    # Copiar para estrutura YOLO
    print("\n3️⃣ Copiando arquivos para estrutura YOLO...")
    
    for split, files in splits.items():
        for img_file, label_file in files:
            # Copiar imagem
            dest_img = dataset_path / 'images' / split / img_file.name
            shutil.copy2(img_file, dest_img)
            
            # Copiar label
            dest_label = dataset_path / 'labels' / split / f"{img_file.stem}.txt"
            shutil.copy2(label_file, dest_label)
    
    # Limpar temp
    shutil.rmtree(temp_labels_dir)
    
    # Relatório
    print("\n" + "="*70)
    print("✅ INTEGRAÇÃO CONCLUÍDA")
    print("="*70)
    print(f"\n📦 Distribuição do dataset:")
    print(f"   Treino:     {len(splits['train']):4} imagens ({len(splits['train'])/len(valid_images)*100:5.1f}%)")
    print(f"   Validação:  {len(splits['val']):4} imagens ({len(splits['val'])/len(valid_images)*100:5.1f}%)")
    print(f"   Teste:      {len(splits['test']):4} imagens ({len(splits['test'])/len(valid_images)*100:5.1f}%)")
    print(f"   TOTAL:      {len(valid_images):4} imagens")
    print("\n" + "="*70)
    
    return dataset_path

# Exemplo de uso (descomente quando pronto):
# dataset = integrate_existing_data(
#     metadata_df,
#     images_dir='C:\\seu\\caminho\\imagens',
#     dataset_path=DATASET_PATH
# )

print("✓ Função de integração legada definida (não usar - usar Célula 5 ao invés)")

✓ Função de integração legada definida (não usar - usar Célula 5 ao invés)


# ⚠️ INTEGRAÇÃO: Use a Célula 5 ao invés disso!

## 🎯 Fluxo Correto

A integração do dataset Lacuna é feita automaticamente na **Célula 5** com a função:
- `prepare_lacuna_to_yolo()` - Converte dataset Lacuna → formato YOLO

Essa célula:
1. ✅ Verifica se o dataset já existe (não sobrescreve)
2. ✅ Lê o CSV com polígonos pré-anotados
3. ✅ Converte polígonos → bounding boxes normalizados
4. ✅ Cria labels no formato YOLO
5. ✅ Organiza em train/val/test automaticamente

## 📍 Próximo Passo

Execute a **Célula 5** para preparar o dataset. Se já tiver sido executada com sucesso, o dataset está pronto em:
```
./data/solar_panels_detection_dataset/
├── images/{train,val,test}/
└── labels/{train,val,test}/
```

Então use a **Célula 6** para preparar e treinar o modelo YOLO.

In [18]:
# ✓ Validação e Relatório do Dataset YOLO
# =========================================

# Após executar Célula 5, você pode validar o resultado aqui

In [19]:
# ========== FUNÇÕES DE VALIDAÇÃO E RELATÓRIO ==========

def validate_yolo_dataset(dataset_path):
    """
    Valida a estrutura do dataset YOLO
    """
    from pathlib import Path
    import os
    
    dataset_path = Path(dataset_path)
    print(f"\n✓ Validando dataset em: {dataset_path}\n")
    
    required_dirs = ['images', 'labels']
    required_splits = ['train', 'val', 'test']
    
    is_valid = True
    
    # Verificar estrutura
    for split in required_splits:
        for dir_name in required_dirs:
            path = dataset_path / split / dir_name
            if path.exists():
                file_count = len(list(path.glob('*')))
                print(f"  ✓ {split}/{dir_name}: {file_count} arquivos")
            else:
                print(f"  ✗ {split}/{dir_name}: NÃO ENCONTRADO")
                is_valid = False
    
    if is_valid:
        print("\n✅ Dataset VÁLIDO!")
    else:
        print("\n⚠️ Dataset INCOMPLETO")
    
    return is_valid


def generate_dataset_report(dataset_path):
    """
    Gera relatório do dataset
    """
    from pathlib import Path
    import os
    
    dataset_path = Path(dataset_path)
    print(f"\n{'='*70}")
    print("📊 RELATÓRIO DO DATASET")
    print(f"{'='*70}\n")
    
    total_images = 0
    total_labels = 0
    splits_info = {}
    
    for split in ['train', 'val', 'test']:
        images_path = dataset_path / split / 'images'
        labels_path = dataset_path / split / 'labels'
        
        images_count = len(list(images_path.glob('*'))) if images_path.exists() else 0
        labels_count = len(list(labels_path.glob('*'))) if labels_path.exists() else 0
        
        splits_info[split] = {'images': images_count, 'labels': labels_count}
        total_images += images_count
        total_labels += labels_count
    
    # Exibir relatório
    print("📈 Distribuição por Split:")
    for split, counts in splits_info.items():
        pct = (counts['images'] / total_images * 100) if total_images > 0 else 0
        print(f"  {split:6}: {counts['images']:6} imagens ({pct:5.1f}%)")
    
    print(f"\n📊 TOTAL: {total_images} imagens, {total_labels} labels")
    print(f"{'='*70}\n")


In [20]:
def explore_metadata_insights(metadata_df):
    """
    Extrai insights dos metadados para análise
    """
    if metadata_df is None:
        return
    
    print("\n" + "="*70)
    print("🔍 INSIGHTS DO DATASET")
    print("="*70 + "\n")
    
    # 1. Análise por origem
    origin_stats = metadata_df['img_origin'].value_counts()
    print("📍 Distribuição por Origem:")
    for origin, count in origin_stats.items():
        pct = count / len(metadata_df) * 100 
        print(f"   {origin:2}: {count:6} imagens ({pct:5.1f}%)")
    
    # 2. Análise por placement
    placement_stats = metadata_df['placement'].value_counts()
    print("\n📦 Distribuição por Localização:")
    for placement, count in placement_stats.items():
        pct = count / len(metadata_df) * 100
        print(f"   {placement:15}: {count:6} imagens ({pct:5.1f}%)")
    
    # 3. Correlação origem x placement
    print("\n🔗 Correlação Origem × Localização:")
    crosstab = pd.crosstab(metadata_df['img_origin'], metadata_df['placement'], margins=True)
    print(crosstab)
    
    # 4. Recomendações
    print("\n💡 RECOMENDAÇÕES:")
    
    total = len(metadata_df)
    roof_count = len(metadata_df[metadata_df['placement'] == 'roof'])
    unknown_count = len(metadata_df[metadata_df['placement'] == 'S-unknown'])
    
    print(f"\n   • Total de amostras: {total}")
    print(f"   • Amostras com rótulo 'roof': {roof_count} ({roof_count/total*100:.1f}%)")
    print(f"   • Amostras 'unknown': {unknown_count} ({unknown_count/total*100:.1f}%)")
    print(f"   • Amostras utilizáveis: {total - unknown_count} ({(total-unknown_count)/total*100:.1f}%)")
    
    if roof_count > total * 0.7:
        print("\n   ✓ Excelente! Dataset bem balanceado em residências")
    elif roof_count > total * 0.5:
        print("\\n   ⚠️  Dataset inclinado para telhados, poucos dados de solo/industrial")
    else:
        print("\\n   ⚠️  Dataset muito desbalanceado, considere fazer augmentation")
    
    print("\n" + "="*70)

# Explorar insights
if metadata_df is not None:
    explore_metadata_insights(metadata_df)


🔍 INSIGHTS DO DATASET

📍 Distribuição por Origem:
   D :    913 imagens ( 82.5%)
   S :    194 imagens ( 17.5%)

📦 Distribuição por Localização:
   roof           :    833 imagens ( 75.2%)
   S-unknown      :    194 imagens ( 17.5%)
   r_openspace    :     40 imagens (  3.6%)
   openspace      :     40 imagens (  3.6%)

🔗 Correlação Origem × Localização:
placement   S-unknown  openspace  r_openspace  roof   All
img_origin                                               
D                   0         40           40   833   913
S                 194          0            0     0   194
All               194         40           40   833  1107

💡 RECOMENDAÇÕES:

   • Total de amostras: 1107
   • Amostras com rótulo 'roof': 833 (75.2%)
   • Amostras 'unknown': 194 (17.5%)
   • Amostras utilizáveis: 913 (82.5%)

   ✓ Excelente! Dataset bem balanceado em residências



## APÊNDICE A: Preparação do Dataset

Guia completo para preparar seu dataset para treinamento YOLO

In [21]:
# PREPARAÇÃO DO DATASET - Estrutura e Organização

# Definir caminho do dataset na pasta notebooks/data
DATASET_NAME = 'solar_panels_detection_dataset'  # Nome do dataset
DATA_BASE_PATH = Path('data')  # Pasta notebooks/data (relativo ao notebook)
DATASET_PATH = DATA_BASE_PATH / DATASET_NAME

print(f"📁 Dataset será criado em: {DATASET_PATH}")
print(f"📁 Caminho absoluto: {DATASET_PATH.resolve()}")

def create_dataset_structure(dataset_path):
    """
    Cria a estrutura padrão YOLO para o dataset
    
    Estrutura criada:
    dataset/
    ├── images/
    │   ├── train/
    │   ├── val/
    │   └── test/
    └── labels/
        ├── train/
        ├── val/
        └── test/
    """
    dataset_path = Path(dataset_path)
    
    # Criar pastas de imagens
    for split in ['train', 'val', 'test']:
        (dataset_path / 'images' / split).mkdir(parents=True, exist_ok=True)
        (dataset_path / 'labels' / split).mkdir(parents=True, exist_ok=True)
    
    print(f"✓ Estrutura de pastas criada em: {dataset_path}")
    print(f"""
    ✓ Diretórios criados:
    ├── images/
    │   ├── train/  (treino)
    │   ├── val/    (validação)
    │   └── test/   (teste)
    └── labels/
        ├── train/  (anotações treino)
        ├── val/    (anotações validação)
        └── test/   (anotações teste)
    """)
    
    return dataset_path

# Criar estrutura
dataset_path = create_dataset_structure(DATASET_PATH)

📁 Dataset será criado em: data/solar_panels_detection_dataset
📁 Caminho absoluto: /home/jovyan/work/data/solar_panels_detection_dataset
✓ Estrutura de pastas criada em: data/solar_panels_detection_dataset

    ✓ Diretórios criados:
    ├── images/
    │   ├── train/  (treino)
    │   ├── val/    (validação)
    │   └── test/   (teste)
    └── labels/
        ├── train/  (anotações treino)
        ├── val/    (anotações validação)
        └── test/   (anotações teste)
    


In [22]:
def organize_images_and_labels(source_images_dir, source_labels_dir, dataset_path, 
                               train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Organiza imagens e labels existentes em estrutura YOLO
    
    ⚠️ IMPORTANTE: Labels devem estar em formato YOLO:
       - Um arquivo .txt por imagem
       - Nome idêntico ao da imagem (ex: image.jpg → image.txt)
       - Formato de cada linha: <class_id> <x_center> <y_center> <width> <height>
       - Todas as coordenadas normalizadas (0-1)
    
    Args:
        source_images_dir: caminho com todas as imagens
        source_labels_dir: caminho com todos os labels (.txt)
        dataset_path: caminho de destino
        train_ratio: proporção treino (0.7 = 70%)
        val_ratio: proporção validação (0.15 = 15%)
        test_ratio: proporção teste (0.15 = 15%)
    """
    import shutil
    from sklearn.model_selection import train_test_split
    
    source_images_dir = Path(source_images_dir)
    source_labels_dir = Path(source_labels_dir)
    dataset_path = Path(dataset_path)
    
    # Verificar diretórios de entrada
    if not source_images_dir.exists():
        print(f"❌ Erro: Diretório de imagens não encontrado: {source_images_dir}")
        return
    
    if not source_labels_dir.exists():
        print(f"❌ Erro: Diretório de labels não encontrado: {source_labels_dir}")
        return
    
    # Listar imagens
    image_files = sorted(
        list(source_images_dir.glob('*.jpg')) + 
        list(source_images_dir.glob('*.png')) +
        list(source_images_dir.glob('*.JPG')) +
        list(source_images_dir.glob('*.PNG'))
    )
    
    print(f"📊 Total de imagens encontradas: {len(image_files)}")
    
    if len(image_files) == 0:
        print("❌ Nenhuma imagem encontrada!")
        return
    
    # Validar proporções
    assert train_ratio + val_ratio + test_ratio == 1.0, "Proporções devem somar 1.0"
    
    # Split dataset
    indices = list(range(len(image_files)))
    train_idx, temp_idx = train_test_split(
        indices, test_size=(1-train_ratio), random_state=42
    )
    val_ratio_adj = val_ratio / (val_ratio + test_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=(1-val_ratio_adj), random_state=42
    )
    
    splits = {
        'train': [image_files[i] for i in train_idx],
        'val': [image_files[i] for i in val_idx],
        'test': [image_files[i] for i in test_idx]
    }
    
    # Copiar arquivos
    copied_count = {'train': 0, 'val': 0, 'test': 0}
    missing_labels = []
    
    for split, files in splits.items():
        for img_file in files:
            # Copiar imagem
            dest_img = dataset_path / 'images' / split / img_file.name
            shutil.copy2(img_file, dest_img)
            
            # Procurar label correspondente
            label_file = source_labels_dir / f"{img_file.stem}.txt"
            if label_file.exists():
                dest_label = dataset_path / 'labels' / split / f"{img_file.stem}.txt"
                shutil.copy2(label_file, dest_label)
                copied_count[split] += 1
            else:
                missing_labels.append(img_file.name)
    
    # Relatório
    print("\n" + "="*60)
    print("✅ ORGANIZAÇÃO CONCLUÍDA")
    print("="*60)
    print(f"📦 Treino:     {copied_count['train']} imagens")
    print(f"📦 Validação:  {copied_count['val']} imagens")
    print(f"📦 Teste:      {copied_count['test']} imagens")
    print(f"📦 TOTAL:      {sum(copied_count.values())} imagens\n")
    
    if missing_labels:
        print(f"⚠️  {len(missing_labels)} labels não encontrados:")
        for label in missing_labels[:5]:
            print(f"   - {label}")
        if len(missing_labels) > 5:
            print(f"   ... e mais {len(missing_labels)-5}")
    
    print("="*60)
    
    return dataset_path

# Exemplo de uso (descomente e ajuste os caminhos):
# dataset = organize_images_and_labels(
#     source_images_dir='C:\\seu\\caminho\\imagens',
#     source_labels_dir='C:\\seu\\caminho\\labels',
#     dataset_path=DATASET_PATH
# )

print("✓ Função de organização de dataset criada")

✓ Função de organização de dataset criada


In [23]:
def validate_yolo_dataset(dataset_path):
    """
    Valida o dataset no formato YOLO
    
    Verifica:
    - Existência de imagens em cada split
    - Correspondência entre imagens e labels
    - Validade do formato dos labels
    """
    dataset_path = Path(dataset_path)
    
    print("\n" + "="*60)
    print("🔍 VALIDANDO DATASET")
    print("="*60 + "\n")
    
    all_valid = True
    
    for split in ['train', 'val', 'test']:
        images_dir = dataset_path / 'images' / split
        labels_dir = dataset_path / 'labels' / split
        
        # Listar arquivos
        image_files = sorted(
            list(images_dir.glob('*.jpg')) + 
            list(images_dir.glob('*.png')) +
            list(images_dir.glob('*.JPG')) +
            list(images_dir.glob('*.PNG'))
        )
        label_files = sorted(list(labels_dir.glob('*.txt')))
        
        print(f"📁 {split.upper()}:")
        print(f"   Imagens: {len(image_files)}")
        print(f"   Labels:  {len(label_files)}")
        
        # Verificar correspondência
        missing_labels = []
        invalid_labels = []
        
        for img_file in image_files:
            label_file = labels_dir / f"{img_file.stem}.txt"
            if not label_file.exists():
                missing_labels.append(img_file.name)
            else:
                # Validar formato
                try:
                    with open(label_file, 'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            parts = line.strip().split()
                            if len(parts) != 5:
                                invalid_labels.append((img_file.name, "Formato incorreto"))
                                break
                            # Verificar se são números
                            try:
                                class_id = int(parts[0])
                                coords = [float(p) for p in parts[1:]]
                                # Verificar range
                                if not all(0 <= c <= 1 for c in coords):
                                    invalid_labels.append((img_file.name, "Coordenadas fora do range [0,1]"))
                            except ValueError:
                                invalid_labels.append((img_file.name, "Valores não numéricos"))
                except Exception as e:
                    invalid_labels.append((img_file.name, str(e)))
        
        if missing_labels:
            print(f"   ❌ Labels ausentes: {len(missing_labels)}")
            all_valid = False
        else:
            print(f"   ✓ Todos os labels presentes")
        
        if invalid_labels:
            print(f"   ❌ Labels inválidos: {len(invalid_labels)}")
            for img, reason in invalid_labels[:3]:
                print(f"      - {img}: {reason}")
            all_valid = False
        else:
            print(f"   ✓ Todos os labels válidos")
        
        print()
    
    if all_valid:
        print("✅ Dataset é válido e pronto para treinamento!")
    else:
        print("⚠️  Dataset possui problemas. Verifique os erros acima.")
    
    print("="*60)
    
    return all_valid

# Validar estrutura criada
validate_yolo_dataset(DATASET_PATH)


🔍 VALIDANDO DATASET

📁 TRAIN:
   Imagens: 2318
   Labels:  2318
   ✓ Todos os labels presentes
   ✓ Todos os labels válidos

📁 VAL:
   Imagens: 496
   Labels:  496
   ✓ Todos os labels presentes
   ✓ Todos os labels válidos

📁 TEST:
   Imagens: 498
   Labels:  498
   ✓ Todos os labels presentes
   ✓ Todos os labels válidos

✅ Dataset é válido e pronto para treinamento!


True

In [24]:
def create_yolo_yaml_config(dataset_path, output_file='data.yaml', dataset_name=None):
    """
    Cria arquivo data.yaml para configuração YOLO
    
    Args:
        dataset_path: caminho absoluto do dataset
        output_file: nome do arquivo de saída
        dataset_name: nome descritivo do dataset
    """
    dataset_path = Path(dataset_path).resolve()
    
    yaml_content = f"""# Dataset Config - Solar Panels Detection
path: {dataset_path}  # dataset root
train: images/train
val: images/val
test: images/test

# Classes
nc: 1  # número de classes
names: ['solar_panel']  # nomes das classes

# Dataset info
info: {dataset_name or 'Solar Panel Detection Dataset'}
"""
    
    output_path = Path(output_file)
    with open(output_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"✓ Arquivo de configuração criado: {output_path}\n")
    print("Conteúdo do arquivo:")
    print("-" * 60)
    print(yaml_content)
    print("-" * 60)
    
    return str(output_path)

# Criar arquivo YAML
yaml_config = create_yolo_yaml_config(
    dataset_path=DATASET_PATH,
    output_file='solar_panels_data.yaml',
    dataset_name='Solar Panel Detection'
)

✓ Arquivo de configuração criado: solar_panels_data.yaml

Conteúdo do arquivo:
------------------------------------------------------------
# Dataset Config - Solar Panels Detection
path: /home/jovyan/work/data/solar_panels_detection_dataset  # dataset root
train: images/train
val: images/val
test: images/test

# Classes
nc: 1  # número de classes
names: ['solar_panel']  # nomes das classes

# Dataset info
info: Solar Panel Detection

------------------------------------------------------------


## APÊNDICE B: Tutorial - Como Preparar seus Dados

### Passo 1: Coletar Imagens
- Tire fotos de telhados com painéis solares
- Use drones, satélites ou imagens de rua
- Variação: diferentes ângulos, iluminações, tamanhos

### Passo 2: Anotar com Formato YOLO
Use ferramentas como:
- **Roboflow**: https://roboflow.com
- **LabelImg**: https://github.com/heartexlabs/labelImg
- **CVAT**: https://cvat.org

Cada imagem precisa de um arquivo `.txt` com:
```
<class_id> <x_center> <y_center> <width> <height>
```

Exemplo (solar_panel.txt):
```
0 0.5 0.5 0.3 0.4
0 0.2 0.7 0.25 0.35
```

### Passo 3: Organizar Estrutura
```python
# Coloque suas imagens em uma pasta
your_images/
├── image1.jpg
├── image1.txt
├── image2.jpg
├── image2.txt
...

# E labels em outra
your_labels/
├── image1.txt
├── image2.txt
...
```

### Passo 4: Executar Organização
```python
# Na célula anterior, descomente e execute:
organize_images_and_labels(
    source_images_dir='caminho/para/imagens',
    source_labels_dir='caminho/para/labels',
    dataset_path=DATASET_PATH
)
```

In [25]:
# EXEMPLO: Criar dataset de exemplo com imagens sintéticas

def create_sample_dataset(dataset_path, num_samples_per_split=5):
    """
    Cria um dataset de exemplo para testes (imagens brancas com padrões aleatórios)
    
    ⚠️ Apenas para testes! Para uso real, use imagens verdadeiras.
    """
    from PIL import Image, ImageDraw
    import random
    
    dataset_path = Path(dataset_path)
    
    print("📸 Criando dataset de exemplo...\n")
    
    for split, num_samples in [('train', num_samples_per_split), 
                                ('val', num_samples_per_split//2), 
                                ('test', num_samples_per_split//2)]:
        
        for idx in range(num_samples):
            # Criar imagem branca
            img = Image.new('RGB', (640, 480), 'white')
            draw = ImageDraw.Draw(img)
            
            # Desenhar painéis solares (azuis)
            num_panels = random.randint(1, 5)
            annotations = []
            
            for _ in range(num_panels):
                # Posição aleatória
                x_center = random.uniform(0.2, 0.8)
                y_center = random.uniform(0.2, 0.8)
                width = random.uniform(0.1, 0.25)
                height = random.uniform(0.1, 0.2)
                
                # Desenhar retângulo (painel)
                x_min = int((x_center - width/2) * 640)
                y_min = int((y_center - height/2) * 480)
                x_max = int((x_center + width/2) * 640)
                y_max = int((y_center + height/2) * 480)
                
                draw.rectangle([x_min, y_min, x_max, y_max], fill='blue', outline='darkblue')
                
                # Adicionar grade (células solares)
                for i in range(4):
                    x_cell = x_min + (x_max - x_min) * i / 4
                    draw.line([(x_cell, y_min), (x_cell, y_max)], fill='black', width=1)
                
                # Annotation YOLO
                annotations.append(f"0 {x_center:.4f} {y_center:.4f} {width:.4f} {height:.4f}")
            
            # Salvar imagem
            img_name = f"sample_{split}_{idx:03d}.jpg"
            img_path = dataset_path / 'images' / split / img_name
            img.save(img_path)
            
            # Salvar labels
            label_path = dataset_path / 'labels' / split / f"sample_{split}_{idx:03d}.txt"
            with open(label_path, 'w') as f:
                f.write('\n'.join(annotations))
        
        print(f"✓ {split.upper()}: {num_samples} imagens de exemplo criadas")
    
    print("\n✅ Dataset de exemplo criado com sucesso!")
    print("⚠️  Nota: Isso é apenas para testes. Use imagens reais para treinamento!")
    
    return dataset_path

# Descomente para criar dataset de exemplo:
# create_sample_dataset(DATASET_PATH, num_samples_per_split=5)

In [26]:
def generate_dataset_report(dataset_path):
    """
    Gera relatório completo do dataset
    """
    dataset_path = Path(dataset_path)
    
    print("\n" + "="*70)
    print("📊 RELATÓRIO DO DATASET")
    print("="*70 + "\n")
    
    print(f"📁 Localização: {dataset_path.resolve()}\n")
    
    total_images = 0
    total_labels = 0
    
    stats = {}
    
    for split in ['train', 'val', 'test']:
        images_dir = dataset_path / 'images' / split
        labels_dir = dataset_path / 'labels' / split
        
        images = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
        labels = list(labels_dir.glob('*.txt'))
        
        total_images += len(images)
        total_labels += len(labels)
        
        # Contar bounding boxes
        total_boxes = 0
        for label_file in labels:
            with open(label_file, 'r') as f:
                total_boxes += len(f.readlines())
        
        stats[split] = {
            'images': len(images),
            'labels': len(labels),
            'boxes': total_boxes
        }
        
        print(f"📦 {split.upper():10} | Imagens: {len(images):4} | Labels: {len(labels):4} | Boxes: {total_boxes:6}")
    
    print(f"\n{'─'*70}")
    print(f"📊 TOTAL      | Imagens: {total_images:4} | Labels: {total_labels:4} | Boxes: {sum(s['boxes'] for s in stats.values()):6}")
    print("="*70 + "\n")
    
    # Proporções
    print("📈 PROPORÇÕES:")
    print(f"   Treino:     {stats['train']['images']/total_images*100:5.1f}%")
    print(f"   Validação:  {stats['val']['images']/total_images*100:5.1f}%")
    print(f"   Teste:      {stats['test']['images']/total_images*100:5.1f}%\n")
    
    # Média de boxes por imagem
    if total_images > 0:
        avg_boxes = sum(s['boxes'] for s in stats.values()) / total_images
        print(f"📊 Média de boxes por imagem: {avg_boxes:.2f}\n")
    
    # Instruções para treinar
    print("🚀 PRÓXIMO PASSO - Para treinar:")
    print(f"""
    config_path = 'solar_panels_data.yaml'
    model = YOLO('yolov8m.pt')
    results = model.train(
        data=config_path,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0
    )
    """)
    print("="*70 + "\n")

# Gerar relatório
generate_dataset_report(DATASET_PATH)


📊 RELATÓRIO DO DATASET

📁 Localização: /home/jovyan/work/data/solar_panels_detection_dataset

📦 TRAIN      | Imagens: 2318 | Labels: 2318 | Boxes:   4687
📦 VAL        | Imagens:  496 | Labels:  496 | Boxes:   1019
📦 TEST       | Imagens:  498 | Labels:  498 | Boxes:   1083

──────────────────────────────────────────────────────────────────────
📊 TOTAL      | Imagens: 3312 | Labels: 3312 | Boxes:   6789

📈 PROPORÇÕES:
   Treino:      70.0%
   Validação:   15.0%
   Teste:       15.0%

📊 Média de boxes por imagem: 2.05

🚀 PRÓXIMO PASSO - Para treinar:

    config_path = 'solar_panels_data.yaml'
    model = YOLO('yolov8m.pt')
    results = model.train(
        data=config_path,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0
    )
    

